In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings(action='ignore')


In [4]:
data = pd.read_csv('delhi.csv')

In [5]:
data


,Area,BHK,Bathroom,Furnishing,Locality,Parking,Price,Status,Transaction,Type,Per_Sqft
0,800.0,3,2.0,Semi-Furnished,Rohini Sector 25,1.0,6500000,Ready_to_move,New_Property,Builder_Floor,NaN
1,750.0,2,2.0,Semi-Furnished,"J R Designers Floors, Rohini Sector 24",1.0,5000000,Ready_to_move,New_Property,Apartment,6667.0
2,950.0,2,2.0,Furnished,"Citizen Apartment, Rohini Sector 13",1.0,15500000,Ready_to_move,Resale,Apartment,6667.0
3,600.0,2,2.0,Semi-Furnished,Rohini Sector 24,1.0,4200000,Ready_to_move,Resale,Builder_Floor,6667.0
4,650.0,2,2.0,Semi-Furnished,Rohini Sector 24 carpet area 650 sqft status R...,1.0,6200000,Ready_to_move,New_Property,Builder_Floor,6667.0
...,...,...,...,...,...,...,...,...,...,...,...
1254,4118.0,4,5.0,Unfurnished,Chittaranjan Park,3.0,55000000,Ready_to_move,New_Property,Builder_Floor,12916.0
1255,1050.0,3,2.0,Semi-Furnished,Chittaranjan Park,3.0,12500000,Ready_to_move,Resale,Builder_Floor,12916.0
1256,875.0,3,3.0,Semi-Furnished,Chittaranjan Park,3.0,17500000,Ready_to_move,New_Property,Builder_Floor,12916.0
1257,990.0,2,2.0,Unfurnished,Chittaranjan Park Block A,1.0,11500000,Ready_to_move,Resale,Builder_Floor,12916.0


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Area         1259 non-null   float64
 1   BHK          1259 non-null   int64  
 2   Bathroom     1257 non-null   float64
 3   Furnishing   1254 non-null   object 
 4   Locality     1259 non-null   object 
 5   Parking      1226 non-null   float64
 6   Price        1259 non-null   int64  
 7   Status       1259 non-null   object 
 8   Transaction  1259 non-null   object 
 9   Type         1254 non-null   object 
 10  Per_Sqft     1018 non-null   float64
dtypes: float64(4), int64(2), object(5)
memory usage: 108.3+ KB


Preprocessing

In [8]:
def onehot_encode(df, column, rename=False):
    df = df.copy()
    if rename == True:
        df[column] = df[column].replace({x: i for i, x in enumerate(df[column].unique())})
    dummies = pd.get_dummies(df[column], prefix=column)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(column, axis=1)
    return df

In [7]:

def preprocess_inputs(df):
    df = df.copy()
    
    # Drop Per_Sqft column
    df = df.drop('Per_Sqft', axis=1)
    
    # Fill missing values
    for column in ['Bathroom', 'Parking', 'Type']:
        df[column] = df[column].fillna(df[column].mode()[0])
    
    # Binary encoding
    df['Status'] = df['Status'].replace({
        'Almost_ready': 0,
        'Ready_to_move': 1
    })
    df['Transaction'] = df['Transaction'].replace({
        'New_Property': 0,
        'Resale': 1
    })
    df['Type'] = df['Type'].replace({
        'Builder_Floor': 0,
        'Apartment': 1
    })
    
    # One-hot encoding
    df = onehot_encode(df, column='Furnishing', rename=False)
    df = onehot_encode(df, column='Locality', rename=True)
    
    # Split df into X and y
    y = df['Price']
    X = df.drop('Price', axis=1)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)
    
    # Scale X
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)
    
    return X_train, X_test, y_train, y_test


In [9]:
X_train, X_test, y_train, y_test = preprocess_inputs(data)

In [11]:
X_train

,Area,BHK,Bathroom,Parking,Status,Transaction,Type,Furnishing_Furnished,Furnishing_Semi-Furnished,Furnishing_Unfurnished,...,Locality_355,Locality_356,Locality_357,Locality_358,Locality_359,Locality_360,Locality_361,Locality_362,Locality_363,Locality_364
582,1.241514,1.245726,1.397555,-0.216906,0.245293,0.779528,-0.963219,-0.412294,0.872926,-0.631199,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
976,0.006189,0.198508,-0.550619,-0.216906,0.245293,0.779528,1.038186,2.425451,-1.145572,-0.631199,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
886,-0.748732,-1.895929,-1.524706,-0.216906,0.245293,-1.282827,1.038186,-0.412294,-1.145572,1.584285,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
561,0.177762,0.198508,0.423468,0.127408,0.245293,0.779528,-0.963219,-0.412294,0.872926,-0.631199,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
1083,-0.587454,-0.848710,-0.550619,-0.216906,0.245293,0.779528,-0.963219,-0.412294,0.872926,-0.631199,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,-0.165384,-0.848710,-0.550619,-0.216906,0.245293,0.779528,1.038186,-0.412294,-1.145572,1.584285,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
905,1.069941,1.245726,1.397555,0.127408,0.245293,-1.282827,-0.963219,-0.412294,-1.145572,1.584285,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
1096,-0.371272,-0.848710,-0.550619,0.127408,0.245293,-1.282827,-0.963219,-0.412294,-1.145572,1.584285,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371
235,0.074818,1.245726,0.423468,-0.216906,0.245293,-1.282827,-0.963219,-0.412294,0.872926,-0.631199,...,-0.058454,-0.03371,0.0,0.0,-0.144421,0.0,-0.03371,-0.03371,-0.03371,-0.03371


In [12]:
y_train

582     55000000
976     14000000
886      1490000
561     30000000
1083     4000000
          ...   
715     14300000
905     67000000
1096     5500000
235     13000000
1061     8000000
Name: Price, Length: 881, dtype: int64

Tranning

In [13]:
models = {
    "                     Linear Regression": LinearRegression(),
    " Linear Regression (L2 Regularization)": Ridge(),
    " Linear Regression (L1 Regularization)": Lasso(),
    "                   K-Nearest Neighbors": KNeighborsRegressor(),
    "                        Neural Network": MLPRegressor(),
    "Support Vector Machine (Linear Kernel)": LinearSVR(),
    "   Support Vector Machine (RBF Kernel)": SVR(),
    "                         Decision Tree": DecisionTreeRegressor(),
    "                         Random Forest": RandomForestRegressor(),
    "                     Gradient Boosting": GradientBoostingRegressor(),
    "                               XGBoost": XGBRegressor(),
    "                              LightGBM": LGBMRegressor(),
    "                              CatBoost": CatBoostRegressor(verbose=0)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")

                     Linear Regression trained.
 Linear Regression (L2 Regularization) trained.
 Linear Regression (L1 Regularization) trained.
                   K-Nearest Neighbors trained.
                        Neural Network trained.
Support Vector Machine (Linear Kernel) trained.
   Support Vector Machine (RBF Kernel) trained.
                         Decision Tree trained.
                         Random Forest trained.
                     Gradient Boosting trained.
                               XGBoost trained.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 170
[LightGBM] [Info] Number of data points in the train set: 881, number of used features: 13
[LightGBM] [Info] Start training from score 20935561.861521
[LightGBM] [Warning] No further splits with positive gain, bes

Result

In [14]:
for name, model in models.items():
    print(name + " R^2 Score: {:.5f}".format(model.score(X_test, y_test)))

                     Linear Regression R^2 Score: 0.67648
 Linear Regression (L2 Regularization) R^2 Score: 0.67672
 Linear Regression (L1 Regularization) R^2 Score: 0.67647
                   K-Nearest Neighbors R^2 Score: 0.59447
                        Neural Network R^2 Score: -0.62240
Support Vector Machine (Linear Kernel) R^2 Score: -0.62248
   Support Vector Machine (RBF Kernel) R^2 Score: -0.07453
                         Decision Tree R^2 Score: 0.69394
                         Random Forest R^2 Score: 0.82702
                     Gradient Boosting R^2 Score: 0.83497
                               XGBoost R^2 Score: 0.87506
                              LightGBM R^2 Score: 0.77550
                              CatBoost R^2 Score: 0.85818


In [15]:
import joblib

# Suppose your best model is called best_model
best_model = CatBoostRegressor()
best_model.fit(X_train, y_train)

# Save the trained model
joblib.dump(best_model, "realty_price_model.pkl")
print("✅ Model saved as realty_price_model.pkl")


Learning rate set to 0.040132
0:	learn: 23832631.3138262	total: 3.72ms	remaining: 3.71s
1:	learn: 23268954.3658457	total: 4.35ms	remaining: 2.17s
2:	learn: 22757665.6771622	total: 4.89ms	remaining: 1.62s
3:	learn: 22333053.5004316	total: 5.41ms	remaining: 1.35s
4:	learn: 21803225.8773525	total: 5.92ms	remaining: 1.18s
5:	learn: 21318570.2308929	total: 6.46ms	remaining: 1.07s
6:	learn: 20884838.2577064	total: 7.07ms	remaining: 1s
7:	learn: 20455454.4812221	total: 7.62ms	remaining: 945ms
8:	learn: 20055311.2093293	total: 8.12ms	remaining: 894ms
9:	learn: 19677197.3193791	total: 8.66ms	remaining: 858ms
10:	learn: 19252608.7083792	total: 9.16ms	remaining: 823ms
11:	learn: 18943672.6058570	total: 9.66ms	remaining: 795ms
12:	learn: 18637506.3328346	total: 10.2ms	remaining: 774ms
13:	learn: 18272745.3832690	total: 10.7ms	remaining: 755ms
14:	learn: 17937163.9004424	total: 11.3ms	remaining: 739ms
15:	learn: 17631882.6124539	total: 11.8ms	remaining: 728ms
16:	learn: 17296702.2618454	total: 13.4